# Reproduce every figure and table — TiRex-2 two-cohort benchmark

**One-shot reproduction, no retraining.** This notebook regenerates *every* figure and table in the
manuscript and supplementary material directly from the pre-computed result files in `results/`
(per-window forecasts, aggregate metric JSONs, and model embeddings). No model is trained or re-run;
the foundation models are never invoked. Runtime is a few minutes on a laptop.

**What you need**
- This repository (or the Zenodo bundle), run from its root.
- A Python env with `numpy pandas scipy matplotlib scikit-learn` (see `requirements.txt`). `umap-learn` is optional (only to recompute UMAP coordinates; the notebook uses the precomputed ones).
- The `results/` folder from the Zenodo data bundle unpacked in place.

**What it does**
1. Regenerates the main figures (Fig 1-6) and supplementary figures, writing them to `outputs/figs/paper/`.
2. Regenerates the explainability figures (UMAP, RSA/CKA).
3. Loads and prints every table's numbers from the canonical `results/tables/*.csv`, for verification
   against the typeset manuscript.

## 0 - Setup: paths, environment check, helper to run a generator

In [ ]:
import os, sys, subprocess, glob, textwrap
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = os.path.abspath(".")
assert os.path.isdir("results"), "Run this notebook from the repository root (results/ must be present)."
FIGDIR = "outputs/figs/paper"; os.makedirs(FIGDIR, exist_ok=True)
PY = sys.executable                     # use the notebook's own interpreter
ENV = dict(os.environ, PYTHONPATH="scripts:datasets/vitaldb:datasets/mover")

def run(script, *args, label=None):
    """Run a figure/table generator as a subprocess; show its tail."""
    cmd = [PY, script, *map(str, args)]
    print(f"$ {' '.join(cmd)}")
    r = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    tail = "\n".join((r.stdout + r.stderr).strip().splitlines()[-6:])
    print(tail)
    if r.returncode != 0:
        raise RuntimeError(f"{script} failed (exit {r.returncode})")
    return r

def show(*names, width=900):
    for n in names:
        p = n if os.path.exists(n) else f"{FIGDIR}/{n}"
        display(Image(filename=p, width=width)) if os.path.exists(p) else print("MISSING:", p)

def show_table(csv_path, caption):
    df = pd.read_csv(csv_path)
    display(Markdown(f"**{caption}**  \n`{csv_path}`"))
    display(df)
    return df

print("Python :", sys.version.split()[0])
# Runtime dependencies for THIS notebook (figures read precomputed results; UMAP coords are
# precomputed in results/_expl_umaps_v2_*.npz, so umap-learn is NOT needed to reproduce figures).
for m in ["numpy","pandas","scipy","matplotlib","sklearn"]:
    try:
        mod=__import__(m); print(f"  {m:12s}", getattr(mod,'__version__','ok'))
    except Exception as e:
        print(f"  {m:12s} MISSING - pip install -r requirements.txt  ({e})")
# umap-learn is only required to RE-COMPUTE embeddings->UMAP from scratch (not part of this notebook).
try:
    import umap; print("  umap        ", umap.__version__, "(optional; only for recomputing UMAP coords)")
except Exception:
    print("  umap         not installed (optional - not needed to reproduce the figures here)")

## 1 - Main + supplementary figures from the forecast pipeline

`scripts/paper_figures.py` regenerates Fig 1-5, the supplementary training-curve figure, and the
supplementary MOVER zero-shot figure, and (re)writes the canonical table files under `results/tables/`.
It reads the aggregate metric JSONs and the per-window forecast CSVs in `results/` - nothing else.

In [ ]:
run("scripts/paper_figures.py", "all2873")

### Figure 1 - Study design and cohort

In [ ]:
show("Fig1_design_cohort.png")

### Figure 2 - Forecast accuracy and the value of the known-future drug covariate

In [ ]:
show("Fig2_accuracy_covariate.png")

### Figure 3 - Zero-shot foundation-model benchmark (headline)

In [ ]:
show("Fig3_zeroshot_tsfm.png")

### Figure 4 - Impending-hypotension prediction vs. task-trained SOTA

In [ ]:
show("Fig4_hypotension_vs_sota.png")

### Figure 5 - Clinical translation and robustness

In [ ]:
show("Fig5_clinical_robustness.png")

### Supplementary - Training curves; MOVER external zero-shot benchmark

In [ ]:
show("FigS_training_curves.png", "FigS_zeroshot_mover.png")

## 2 - Figure 6 (cross-dataset transfer) and the decision-curve supplementary figure
`transfer_figure.py` also (re)writes Table 7; `decision_curves_figure.py` produces the DCA figure.

In [ ]:
run("scripts/transfer_figure.py")
show("Fig6_transfer.png")

In [ ]:
run("scripts/decision_curves_figure.py")
show("FigS_decision_curves.png")

## 3 - Explainability figures (representation analysis)
UMAP of per-window encoder representations (both cohorts) and the cross-model RSA/CKA heatmap.
These read the model embeddings (`results/embeddings_*.npz`, `results/_expl_umaps_v2_*.npz`) and the
canonical probe CSV (`results/suppl_probe_hypotension.csv`) for the per-panel AUROC labels.

In [ ]:
run("scripts/explainability/fig_umap_final.py")
run("scripts/explainability/fig_rsa.py")
# these write to the project root; show from there
show("FigS_umap_vitaldb.png", "FigS_umap_mover.png")
show("FigS_rsa_cka.png")

## 4 - Tables: regenerate and print every table's numbers

The canonical table values live in `results/tables/*.csv`, regenerated above by the figure/stats
scripts straight from the result files. Two generators produce the remaining tables:

In [ ]:
run("scripts/external_table.py")
run("scripts/stats_tests.py")

### Main tables 1-8

In [ ]:
_ = show_table("results/tables/Table1_cohort.csv",            "Table 1 - Cohort characteristics")

In [ ]:
_ = show_table("results/tables/Table2_accuracy.csv",          "Table 2 - Forecast accuracy")

In [ ]:
_ = show_table("results/tables/Table3_classification.csv",    "Table 3 - Hypotension classification vs. comparators")

In [ ]:
_ = show_table("results/tables/Table4_matched.csv",           "Table 4 - Matched vs. task-trained")

In [ ]:
_ = show_table("results/tables/Table5_matched_forecast.csv",  "Table 5 - Matched forecast accuracy")

In [ ]:
_ = show_table("results/tables/Table6_zeroshot.csv",          "Table 6 - Zero-shot foundation-model benchmark")

In [ ]:
_ = show_table("results/tables/Table7_transfer.csv",          "Table 7 - Cross-dataset transfer")

In [ ]:
_ = show_table("results/tables/Table8_external.csv",          "Table 8 - External validation (MOVER)")

### Supplementary tables (statistics; MOVER zero-shot; explainability)

In [ ]:
_ = show_table("results/tables/TableS_stats.csv",             "Table S - Paired significance tests")

In [ ]:
_ = show_table("results/tables/TableS_zeroshot_mover.csv",    "Table S - MOVER zero-shot benchmark")

The four explainability supplementary tables are provided as machine-readable CSVs
(regenerated by `scripts/explainability/expl_consolidate.py`):

In [ ]:
for f in ["suppl_probe_hypotension","suppl_probe_trajectory","suppl_cluster_characterization","suppl_rsa_cka_VitalDB","suppl_rsa_cka_MOVER"]:
    p=f"results/{f}.csv"
    if os.path.exists(p): _ = show_table(p, f)

## 5 - Figure/table -> result-file provenance map
For auditability: which result file(s) supply each deliverable.

In [ ]:
prov = {
 "Fig 1 design/cohort": "ablation_primary_all2873.json, cohort_flow.json, ablation_windows_all2873.csv (example panels)",
 "Fig 2 accuracy+covariate": "ablation_primary_all2873.json, ablation_primary_all2873_covrate.json",
 "Fig 3 zero-shot benchmark": "ablation_primary_all2873.json, hypo_metrics_all2873.json, ablation_windows_baseline-*_all2873.csv",
 "Fig 4 hypotension vs SOTA": "hypo_metrics_all2873.json, hypo_metrics_baseline-*_all2873.json, clinical_eval_*",
 "Fig 5 clinical robustness": "clinical_eval_all2873.json, subgroup_forest_all2873_h5.json, subgroup_forest_mover_art_h5.json",
 "Fig 6 transfer": "ablation_windows_xfer-*_*.csv, ablation_windows_{all2873,mover_art}.csv",
 "FigS training curves": "baseline_history_* (via ablation_primary)",
 "FigS MOVER zero-shot": "ablation_primary_mover_art.json, hypo_metrics_mover_art.json",
 "FigS decision curves": "ablation_windows_{all2873,mover_art}.csv (+ optional clinical_data.csv for subject split)",
 "FigS UMAP (both cohorts)": "_expl_umaps_v2_{all2873,mover_art}.npz, suppl_probe_hypotension.csv",
 "FigS RSA/CKA": "_expl_rsa_both.json (from embeddings_*.npz)",
 "Tables 1-6, S(zeroshot)": "results/tables/*.csv (paper_figures.py)",
 "Table 7": "results/tables/Table7_transfer.csv (transfer_figure.py)",
 "Table 8": "results/tables/Table8_external.csv (external_table.py)",
 "Table S (stats)": "results/tables/TableS_stats.csv (stats_tests.py)",
 "Suppl. explainability tables": "results/suppl_*.csv (expl_consolidate.py)",
}
display(pd.DataFrame([(k,v) for k,v in prov.items()], columns=["Deliverable","Result file(s)"]))
print("\nAll deliverables regenerated with no retraining and no foundation-model inference.")